## Setup & Secrets

In [1]:
import os
import hmac
import hashlib
import duckdb
import pandas as pd
from dotenv import load_dotenv

os.makedirs(os.path.join("data", "Hashed"), exist_ok=True)

# Removes old cache
load_dotenv(override=True)

# get key version
version_tag = "V2"
PEPPER = os.getenv(f"APP_PEPPER_{version_tag}")

if not PEPPER:
    raise ValueError(f"APP_PEPPER_{version_tag} not found in .env file.")

print(f"Active key: APP_PEPPER_{version_tag}")



Active key: APP_PEPPER_V2


## Transformation & Hashing

In [2]:


source_file_path = os.path.join("data", "Cleaned", "cleaned_records.parquet")
os.makedirs(os.path.join("data", "Transformed"), exist_ok=True)
transformed_file_path = os.path.join("data", "Transformed", "Transformed.parquet")
hashed_file_path = os.path.join("data", "Hashed", "Hashed.parquet")

# Load data
df = duckdb.sql(f"SELECT * FROM '{source_file_path}'").df()

# STEP 1: Create Transformed data (WITHOUT hash columns) ---
query = f"""
WITH price_avgs AS (
    SELECT 
        *,
        AVG(resale_price) OVER (PARTITION BY month, town, flat_type) AS avg_price_group
    FROM df
),
processed_parts AS (
    SELECT 
        *,
        'S' AS p1,
        LPAD(LEFT(REGEXP_REPLACE(block, '[^0-9]', '', 'g'), 3), 3, '0') AS p2,
        SUBSTR(CAST(CAST(ROUND(avg_price_group) AS BIGINT) AS VARCHAR), 1, 2) AS p3,
        SUBSTR(month, 6, 2) AS p4,
        UPPER(SUBSTR(town, 1, 1)) AS p5
    FROM price_avgs
)
SELECT 
    * EXCLUDE (p1, p2, p3, p4, p5),
    p1 || p2 || p3 || p4 || p5 AS resale_identifier
FROM processed_parts
"""

transformed_df = duckdb.sql(query).df()
transformed_df = transformed_df.loc[:, ~transformed_df.columns.duplicated()]

# Save baseline transformations to Transformed.parquet
transformed_df.to_parquet(transformed_file_path, index=False)
print(f"Successfully generated: {transformed_file_path}")

#  STEP 2: Create Hash using SHA 256 with PEPPER
def secure_hash(code):
    return hmac.new(PEPPER.encode('utf-8'), str(code).encode('utf-8'), hashlib.sha256).hexdigest()

hashed_df = duckdb.sql(f"SELECT * FROM '{transformed_file_path}'").df()

# Append the hash and version tracking columns
hashed_df['hashed_resale_identifier'] = hashed_df['resale_identifier'].apply(secure_hash)
hashed_df['hash_version'] = version_tag

# Save final result to Hashed parquet
hashed_df.to_parquet(hashed_file_path, index=False)
print(f"Successfully generated: {hashed_file_path}")

print(hashed_df[['resale_identifier', 'hash_version', 'hashed_resale_identifier']].head(5))

Successfully generated: data\Transformed\Transformed.parquet


Successfully generated: data\Hashed\Hashed.parquet
  resale_identifier hash_version  \
0         S4725306P           V2   
1         S1895306P           V2   
2         S1255306P           V2   
3         S5535306P           V2   
4         S7425306P           V2   

                            hashed_resale_identifier  
0  7f4d216fd3060691bb5445f5b1d4275e1cff82b4fb410b...  
1  0e0d123bd135961ee95d3a256aede1c7b8da749e98d3a2...  
2  f56828d18a298087f780dbfd1a04988c08c033a6829c03...  
3  b75c209ca44e9dd6bdbc332e9a88165d929923dbe35cc5...  
4  8b251bb079d37d01c2f530689bd6e1df92634df7b4129d...  
